# ALPR - JL

Just run "Image clarity intergration" cell and "Compare and Aggregate OCR across frames"

If more improvment requried

Number Plate Detetion:

https://github.com/bharatsubedi/ALPR-Yolov5/tree/master

https://github.com/ashok426/Vehicle-number-plate-recognition-YOLOv5/blob/main/License_plate_extraction_yolo.ipynb

OCR:

Darknet: https://youtu.be/jz97_-PCxl4

## Requirments

In [ ]:
!python3 -m venv "envALPR"

In [ ]:
!source envALPR/bin/activate

In [ ]:
!pip install opencv-python torch torchvision # if using YOLOv5 or YOLOv8 with PyTorch

In [ ]:
!git clone https://github.com/ultralytics/yolov5.git

In [ ]:
!pip install -r "yolov5/requirements.txt"

In [ ]:
!git clone https://github.com/ZQPei/deep_sort_pytorch.git

In [ ]:
!pip install -r "deep_sort_pytorch/requirements.txt"

In [ ]:
!cd deep_sort_pytorch/detector/YOLOv5

In [ ]:
!wget https://github.com/ultralytics/yolov5/releases/download/v6.1/yolov5s.pt

In [ ]:
# if you use original model in PAPER
!cd deep_sort/deep/checkpoint
# download ckpt.t7 from
https://drive.google.com/drive/folders/1xhG0kRH1EX5B9_Iz8gQJb7UNnn_riXi6 to this folder
!cd ../../../

In [ ]:
!mkdir "EasyOCR"

## Load Yolov8

In [ ]:
import warnings

# Suppress FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import cv2
import torch

# Load the YOLO model (e.g., YOLOv5 or YOLOv8)
model = torch.hub.load('yolov5', 'yolov5s', pretrained=True, source='local')

# Adjust classes to focus on vehicles only
# You might need to specify vehicle class IDs based on the model you use
vehicle_classes = [2, 3, 5, 7]  # Example class IDs for car, motorcycle, bus, and truck

video_source = "data/1_truck_1920x1080.mp4"
# video_source = "data/1_van.mp4"
# video_source = "data/test3.mp4"
cap = cv2.VideoCapture(video_source)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Couldn't read frame")
        break

    # Run YOLO detection
    results = model(frame)
    detections = results.pred[0].numpy()  # Assuming YOLOv5 format

    # Filter out non-vehicle detections
    vehicle_detections = [d for d in detections if int(d[5]) in vehicle_classes]

    # Draw bounding boxes
    for x1, y1, x2, y2, conf, cls_id in vehicle_detections:
        label = f"{model.names[int(cls_id)]} {conf:.2f}"
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(frame, label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display frame with detections
    cv2.imshow("Vehicle Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


## DeepSort Intergration

In [ ]:
import warnings

# Suppress FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import cv2
import torch
from deep_sort_pytorch.deep_sort import DeepSort  # Import DeepSORT

# Load YOLO model
model = torch.hub.load('yolov5', 'yolov5s', pretrained=True, source='local')
vehicle_classes = [2, 3, 5, 7]  # Class IDs for car, motorcycle, bus, and truck

# Initialize DeepSORT
deepsort = DeepSort(model_path="deep_sort_pytorch/deep_sort/deep/checkpoint/ckpt.t7")  # Use path to your DeepSORT model

# Video source
video_source = "data/1_truck_1920x1080.mp4"
# video_source = "data/1_van.mp4"
# video_source = "data/test3.mp4"
cap = cv2.VideoCapture(video_source)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Couldn't read frame")
        break

    # Run YOLO detection
    results = model(frame)
    detections = results.pred[0].numpy()  # YOLOv5 format

    # Filter out non-vehicle detections
    vehicle_detections = [d for d in detections if int(d[5]) in vehicle_classes]

    # Prepare DeepSORT input
    bbox_xywh = []
    confidences = []
    for x1, y1, x2, y2, conf, cls_id in vehicle_detections:
        bbox = [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]  # x_center, y_center, width, height
        bbox_xywh.append(bbox)
        confidences.append(conf)

    # Perform tracking
    outputs = deepsort.update(bbox_xywh, confidences, cls_id, ori_img=frame)

    # Draw results
    for output in outputs:
        x1, y1, x2, y2, obj_id = output[:5]
        label = f"ID {obj_id}"
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(frame, label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display frame with tracking
    cv2.imshow("Vehicle Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import warnings

# Suppress FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import cv2
import torch
import numpy as np
from deep_sort_pytorch.deep_sort import DeepSort  # Import DeepSORT

# Load YOLO model
model = torch.hub.load('yolov5', 'yolov5s', pretrained=True, source='local')
vehicle_classes = [2, 3, 5, 7]  # Class IDs for car, motorcycle, bus, and truck

# Initialize DeepSORT
deepsort = DeepSort(model_path="deep_sort_pytorch/deep_sort/deep/checkpoint/ckpt.t7")  # Use path to your DeepSORT model

# Video source
video_source = "data/1_truck_1920x1080.mp4"
# video_source = "data/1_van.mp4"
# video_source = "data/test3.mp4"
cap = cv2.VideoCapture(video_source)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Couldn't read frame")
        break

    # Run YOLO detection
    results = model(frame)
    detections = results.pred[0].numpy()  # YOLOv5 format

    # Filter out non-vehicle detections
    vehicle_detections = [d for d in detections if int(d[5]) in vehicle_classes]
    
    # Debug: Print the number of vehicle detections
    print(f"Vehicle detections: {len(vehicle_detections)}")

    # Prepare DeepSORT input
    bbox_xywh = []
    confidences = []
    classes = []
    
    for x1, y1, x2, y2, conf, cls_id in vehicle_detections:
        bbox = [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]
        bbox_xywh.append(bbox)
        confidences.append(conf)
        classes.append(int(cls_id))

        # Debug: Print each detection's confidence
        print(f"Detection - Conf: {conf}, Class ID: {cls_id}")

    # Convert bbox_xywh to a NumPy array
    bbox_xywh = np.array(bbox_xywh)

    # Perform tracking
    outputs = deepsort.update(bbox_xywh, confidences, classes, ori_img=frame)

    # Debug: Print outputs from DeepSORT
    print(f"Outputs from DeepSORT: {outputs}")

    # Draw results
    for output in outputs[0]:  # Only process the first output
        if len(output) < 6:
            print("Warning: output does not contain enough values")
            continue
        x1, y1, x2, y2, cls, obj_id = output
        label = f"Class {cls} | ID {obj_id}"
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(frame, label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display frame with tracking
    cv2.imshow("Vehicle Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


## Image clarity intergration

In [ ]:
import warnings
import cv2
import torch
import numpy as np
from deep_sort_pytorch.deep_sort import DeepSort

# Suppress FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Load YOLO model
model = torch.hub.load('yolov5', 'yolov5s', pretrained=True, source='local')
vehicle_classes = [2, 3, 5, 7]  # Class IDs for car, motorcycle, bus, and truck

# Initialize DeepSORT
deepsort = DeepSort(model_path="deep_sort_pytorch/deep_sort/deep/checkpoint/ckpt.t7")

# Video source
# video_source = "data/1_truck_1920x1080.mp4"
# video_source = "data/1_van.mp4"
video_source = "data/multi_truck_4k.mp4"
# video_source = "data/test3.mp4"
cap = cv2.VideoCapture(video_source)

# Dictionary to store the sharpest frame for each track_id
best_frames = {}

def calculate_sharpness(frame):
    """Calculate the sharpness of the frame using Laplacian variance."""
    print("calculating sharpness...")
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Couldn't read frame")
        break

    # Run YOLO detection
    results = model(frame)
    detections = results.pred[0].numpy()

    # Filter out non-vehicle detections
    vehicle_detections = [d for d in detections if int(d[5]) in vehicle_classes]

    # Prepare DeepSORT input
    bbox_xywh = []
    confidences = []
    classes = []
    
    for x1, y1, x2, y2, conf, cls_id in vehicle_detections:
        bbox = [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]
        bbox_xywh.append(bbox)
        confidences.append(conf)
        classes.append(int(cls_id))

    bbox_xywh = np.array(bbox_xywh)

    # Perform tracking
    outputs = deepsort.update(bbox_xywh, confidences, classes, ori_img=frame)

    # Process each tracked vehicle
    for output in outputs[0]:  # Only process the first output
        if len(output) < 6:
            print("Warning: output does not contain enough values")
            continue
        x1, y1, x2, y2, cls, obj_id = output
    # for output in outputs:
    #     if len(output) < 6:
    #         continue
    #     x1, y1, x2, y2, cls, obj_id = map(int, output[:6])

        # Crop the vehicle region
        vehicle_img = frame[y1:y2, x1:x2]

        # Calculate the sharpness of the cropped image
        sharpness = calculate_sharpness(vehicle_img) 

        print(sharpness) ## Debug

        # Check if this is the clearest frame for this vehicle (track_id)
        if obj_id not in best_frames or sharpness > best_frames[obj_id]["sharpness"]:
            best_frames[obj_id] = {
                "frame": vehicle_img,
                "sharpness": sharpness
            }

        # Draw bounding boxes and track ID on the frame
        label = f"ID {obj_id}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display the tracking results
    cv2.imshow("Vehicle Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Save the best frames to disk after the video ends
for obj_id, data in best_frames.items():
    filename = f"vehicle_{obj_id}.jpg"
    cv2.imwrite(filename, data["frame"])
    print(f"Saved best frame for vehicle ID {obj_id} as {filename}")

cap.release()
cv2.destroyAllWindows()


## Number plate reconation and OCR

In [ ]:
import easyocr

# Initialize the EasyOCR reader
reader = easyocr.Reader(lang_list=['en'], model_storage_directory='EasyOCR/')  # Specify the languages you want to use

def preprocess_license_plate(license_plate_img):
    # Convert to grayscale
    gray = cv2.cvtColor(license_plate_img, cv2.COLOR_BGR2GRAY)
    
    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Enhance contrast
    enhanced = cv2.convertScaleAbs(blurred, alpha=2, beta=0)  # Adjust alpha and beta as needed

    # Apply adaptive thresholding for binarization
    binary_img = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                       cv2.THRESH_BINARY, 11, 2)

    # Optional: Edge enhancement with Canny edge detection
    edges = cv2.Canny(binary_img, 100, 200)
    
    return binary_img, edges

# Perform OCR on the preprocessed license plate
def ocr_license_plate(license_plate_img):
    # Use EasyOCR to read text from the image
    results = reader.readtext(license_plate_img, detail=0)
    return " ".join(results)  # Join the results into a single string

# Assuming you have a trained YOLO model for license plates
license_plate_model = torch.hub.load('yolov5', 'custom', path='priyammehta_best.pt', source='local')

for obj_id, data in best_frames.items():
    vehicle_img = data["frame"]
    
    # Run license plate detection on the vehicle image
    results = license_plate_model(vehicle_img)
    lp_detections = results.pred[0].numpy()  # Adjust if using a different model

    # Ensure there is at least one detection
    if lp_detections.size > 0:
        # Get the coordinates of the first license plate detected
        x1, y1, x2, y2, conf, cls_id = lp_detections[0]
        license_plate_img = vehicle_img[int(y1):int(y2), int(x1):int(x2)]

        # Preprocess the license plate image for OCR
        binary_img, edges = preprocess_license_plate(license_plate_img) 

        # Perform OCR using EasyOCR with actual plate
        plate_text_raw = ocr_license_plate(license_plate_img)
        print(f"Actual: Vehicle ID {obj_id} License Plate: {plate_text}")

        # Perform OCR using EasyOCR with preprocessed plate
        plate_text_processed = ocr_license_plate(binary_img)
        print(f"Processed: Vehicle ID {obj_id} License Plate: {plate_text}")
        
        # Optional: Save the detected license plate region for verification
        cv2.imwrite(f"license_plate_{obj_id}.jpg", license_plate_img)
        cv2.imwrite(f"processed_license_plate_{obj_id}.jpg", binary_img)
        
        # Now proceed with OCR on this `license_plate_img`
    else:
        print(f"No license plate detected for vehicle ID {obj_id}")


## Compare and Aggregate OCR Results Across Frames

In [ ]:
import re
import easyocr
from collections import defaultdict, Counter

# Dictionary to store OCR results for each vehicle
ocr_results = defaultdict(list)

# Initialize the EasyOCR reader
reader = easyocr.Reader(lang_list=['en'], model_storage_directory='EasyOCR/')  # Specify the languages you want to use

def preprocess_license_plate(license_plate_img):
    # Convert to grayscale
    gray = cv2.cvtColor(license_plate_img, cv2.COLOR_BGR2GRAY)
    
    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Enhance contrast
    enhanced = cv2.convertScaleAbs(blurred, alpha=2, beta=0)  # Adjust alpha and beta as needed

    # Apply adaptive thresholding for binarization
    binary_img = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                       cv2.THRESH_BINARY, 11, 2)

    # Optional: Edge enhancement with Canny edge detection
    edges = cv2.Canny(binary_img, 100, 200)
    
    return binary_img, edges

# Perform OCR on the preprocessed license plate
def ocr_license_plate(license_plate_img):
    # Use EasyOCR to read text from the image
    results = reader.readtext(license_plate_img, detail=0)
    return " ".join(results)  # Join the results into a single string

def validate_plate_text(text):
    # Customize regex based on your expected license plate formats
    plate_pattern = r"^[A-Z0-9]{6,8}$"  # Example pattern: Alphanumeric, 6-8 characters(Indina number plates may have 10-12)
    return bool(re.match(plate_pattern, text))

# Assuming you have a trained YOLO model for license plates
license_plate_model = torch.hub.load('yolov5', 'custom', path='priyammehta_best.pt', source='local')

for obj_id, data in best_frames.items():
    vehicle_img = data["frame"]
    
    # Run license plate detection on the vehicle image
    results = license_plate_model(vehicle_img)
    lp_detections = results.pred[0].numpy()  # Adjust if using a different model

    # Ensure there is at least one detection
    if lp_detections.size > 0:
        # Get the coordinates of the first license plate detected
        x1, y1, x2, y2, conf, cls_id = lp_detections[0]
        license_plate_img = vehicle_img[int(y1):int(y2), int(x1):int(x2)]

        # Preprocess the license plate image for OCR
        binary_img, edges = preprocess_license_plate(license_plate_img) 

        # Perform OCR using EasyOCR with actual plate
        plate_text_raw = ocr_license_plate(license_plate_img)
        print(f"Actual: Vehicle ID {obj_id} License Plate: {plate_text_raw}")

        # Perform OCR using EasyOCR with preprocessed plate
        plate_text_processed = ocr_license_plate(binary_img)
        print(f"Processed: Vehicle ID {obj_id} License Plate: {plate_text_processed}")

        # Choose the most confident OCR result or add both if uncertain
        ocr_results[obj_id].append(plate_text_processed if plate_text_processed else plate_text_raw)
        # After processing, aggregate results
        for obj_id, texts in ocr_results.items():
            valid_texts = [text for text in texts if validate_plate_text(text)]
            if valid_texts:
                # Count occurrences of each OCR result
                most_common_text = Counter(texts).most_common(1)[0][0]
                print('-'*5)
                print(f"Final License Plate for Vehicle ID {obj_id}: {most_common_text}")
                print('-'*5)
            else:
                 print(f"No valid license plate detected for Vehicle ID {obj_id}")
        
        # Optional: Save the detected license plate region for verification
        cv2.imwrite(f"license_plate_{obj_id}.jpg", license_plate_img)
        cv2.imwrite(f"processed_license_plate_{obj_id}.jpg", binary_img)
        
        # Now proceed with OCR on this `license_plate_img`
    else:
        print(f"No license plate detected for vehicle ID {obj_id}")
